# ZelusBench: Attention Shifting — Adapt & Recompute

Can the model update its mental model when transforms invalidate prior computation?

**Independent variable**: transform density (transform_prob)
- Transform probs: [0.0, 0.1, 0.2, 0.3, 0.5]
- At 0.0, pure chain-following. At 0.5, ~half the steps are rotations/reflections/translations
- Transforms cascade: rotating point D invalidates everything downstream
- depth=6, num_points=12, dim=3 fixed
- POSITION queries placed after transforms, query_min_depth=4
- 10 seeds per level = 50 scenarios

In [ ]:
import kaggle_benchmarks as kbench
import numpy as np
import random
import time

from zelusbench.scenarios.config import ScenarioConfig, QueryType
from zelusbench.scenarios.generator import ScenarioGenerator
from zelusbench.evaluation.scorer import score_query, score_response

DEPTH = 6
NUM_POINTS = 12
TRANSFORM_PROBS = [0.0, 0.1, 0.2, 0.3, 0.5]
SEEDS = 10
TOTAL = len(TRANSFORM_PROBS) * SEEDS
print(f"ZelusBench Attention Shifting — Adapt & Recompute")
print(f"Depth: {DEPTH} | Points: {NUM_POINTS} | Transform probs: {TRANSFORM_PROBS}")
print(f"Seeds: {SEEDS} | Total: {TOTAL} scenarios")

In [ ]:
@kbench.task(name="zelusbench_shifting")
def zelusbench_shifting(llm) -> tuple[float, float]:
    _start = time.time()
    print(f"Running {TOTAL} scenarios...")
    print("=" * 60)
    all_scenario_avgs = []
    level_scores = {}
    scenario_num = 0
    for tp in TRANSFORM_PROBS:
        level_scores[tp] = []
        for i in range(SEEDS):
            scenario_num += 1
            bg_rng = random.Random(i * 7919)
            cfg = ScenarioConfig.randomize_except(bg_rng, pinned={
                "dim": 3,
                "min_chain_depth": DEPTH, "max_chain_depth": DEPTH,
                "num_points": NUM_POINTS,
                "transform_prob": tp,
                "query_types": [QueryType.POSITION],
                "query_min_depth": 4,
                "num_queries": 3,
                "seed": int(tp * 1000) + i,
            })
            scenario = ScenarioGenerator(cfg).generate(f"shifting_{tp}_{i}")
            response = llm.prompt(scenario.prompt)
            scores = score_response(response, scenario)
            avg = float(np.mean([s.score for s in scores]))
            level_scores[tp].append(avg)
            all_scenario_avgs.append(avg)
            print(f"  [{scenario_num}/{TOTAL}] tp={tp} avg={avg:.2f} | lb={cfg.leaf_bias}")
        print(f"  >> Transform prob {tp} mean: {float(np.mean(level_scores[tp])):.3f}")
    print("\n" + "=" * 60)
    for tp in TRANSFORM_PROBS:
        avg = round(float(np.mean(level_scores[tp])), 3)
        sem = round(float(np.std(level_scores[tp]) / np.sqrt(len(level_scores[tp]))), 3)
        print(f"  tp={tp:.1f}  accuracy={avg:.3f} +/- {sem:.3f}")
        kbench.assertions.assert_true(
            avg > 0.0,
            expectation=f"Shifting tp={tp} (transforms): accuracy={avg:.3f} +/- {sem:.3f}"
        )
    score = round(float(np.mean(all_scenario_avgs)), 3)
    confidence = round(float(np.std(all_scenario_avgs) / np.sqrt(len(all_scenario_avgs))), 3)
    print(f"\nScore: {score:.3f} +/- {confidence:.3f} (SEM) | {len(all_scenario_avgs)} scenarios | {time.time()-_start:.1f}s")
    return score, confidence

zelusbench_shifting.run(llm=kbench.llm)

In [ ]:
%choose zelusbench_shifting